In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [3]:
file_path = "Day12_Used_Car_Preprocessing_Dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset Shape: (320, 15)


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


In [4]:
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

Number of Rows: 320
Number of Columns: 15

Column Names:
['Car_ID', 'Brand', 'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type', 'Condition', 'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Resale_Price_Lakh']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    object 
 1   Brand               320 non-null    object 
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    object 
 7   Transmission        320 non-null    object 
 8   City                320 non-null    object 
 9   Seller_Type         320 non-null    object 
 10  Condition  

In [5]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Car_ID,320,320,CAR0304,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Brand,320,10,Volkswagen,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year,320.0,NaN,NaN,NaN,2019.5375,3.341367,2014.0,2017.0,2020.0,2022.0,2025.0
Mileage_Km,320.0,NaN,NaN,NaN,74110.203125,38885.260771,700.0,46323.25,72718.5,97951.5,320000.0
Engine_CC,320.0,NaN,NaN,NaN,1346.703125,543.40816,600.0,1004.75,1303.0,1635.25,5000.0
Power_BHP,320.0,NaN,NaN,NaN,150.489688,36.665353,51.4,128.45,150.75,171.475,390.0
Fuel_Type,320,4,Petrol,182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Transmission,320,2,Manual,197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,320,10,Lucknow,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Seller_Type,320,3,Individual,167,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
missing_report = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

display(missing_report)

,Missing Values,Missing Percentage
Car_ID,0,0.0
Brand,0,0.0
Year,0,0.0
Mileage_Km,0,0.0
Engine_CC,0,0.0
Power_BHP,0,0.0
Fuel_Type,0,0.0
Transmission,0,0.0
City,0,0.0
Seller_Type,0,0.0


In [7]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 0


In [8]:
numeric_columns = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Previous_Owners",
    "Accidents_Reported",
    "Service_Score",
    "Resale_Price_Lakh"
]

print("Numeric Columns:")
print(numeric_columns)

Numeric Columns:
['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Resale_Price_Lakh']


In [9]:
outlier_report = []

for column in numeric_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    outlier_report.append({
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": len(outliers)
    })

outlier_report = pd.DataFrame(outlier_report)

display(outlier_report)

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Year,2017.0000,2022.000,5.0000,2009.50000,2029.50000,0
1,Mileage_Km,46323.2500,97951.500,51628.2500,-31119.12500,175393.87500,2
2,Engine_CC,1004.7500,1635.250,630.5000,59.00000,2581.00000,6
3,Power_BHP,128.4500,171.475,43.0250,63.91250,236.01250,7
4,Previous_Owners,1.0000,2.000,1.0000,-0.50000,3.50000,14
5,Accidents_Reported,0.0000,0.000,0.0000,0.00000,0.00000,63
6,Service_Score,64.7500,87.000,22.2500,31.37500,120.37500,0
7,Resale_Price_Lakh,2.2775,6.835,4.5575,-4.55875,13.67125,5


In [10]:
# Car_ID is an identifier and does not provide useful predictive information

X = df.drop(columns=["Car_ID", "Resale_Price_Lakh"])

y = df["Resale_Price_Lakh"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Features Shape: (320, 13)
Target Shape: (320,)

Features:
['Brand', 'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type', 'Condition', 'Previous_Owners', 'Accidents_Reported', 'Service_Score']

Target:
Resale_Price_Lakh


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

print("Training Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

Training Data Shape: (256, 13)
Testing Data Shape: (64, 13)
Training Target Shape: (256,)
Testing Target Shape: (64,)


In [12]:
def calculate_iqr_bounds(data, columns):

    bounds = {}

    for column in columns:

        Q1 = data[column].quantile(0.25)
        Q3 = data[column].quantile(0.75)

        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        bounds[column] = {
            "lower": lower_bound,
            "upper": upper_bound
        }

    return bounds

In [13]:
continuous_columns = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Service_Score"
]

print("Continuous columns for IQR treatment:")
print(continuous_columns)

Continuous columns for IQR treatment:
['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Service_Score']


In [14]:
iqr_bounds = calculate_iqr_bounds(
    X_train,
    continuous_columns
)

for column, bounds in iqr_bounds.items():

    print(
        f"{column}: "
        f"Lower = {bounds['lower']:.2f}, "
        f"Upper = {bounds['upper']:.2f}"
    )

Year: Lower = 2009.50, Upper = 2029.50
Mileage_Km: Lower = -31351.75, Upper = 175124.25
Engine_CC: Lower = 66.00, Upper = 2592.00
Power_BHP: Lower = 67.46, Upper = 233.36
Service_Score: Lower = 35.38, Upper = 116.38


In [15]:
def apply_iqr_capping(data, bounds):

    data = data.copy()

    for column, limits in bounds.items():

        data[column] = data[column].clip(
            lower=limits["lower"],
            upper=limits["upper"]
        )

    return data

In [16]:
X_train_capped = apply_iqr_capping(
    X_train,
    iqr_bounds
)

X_test_capped = apply_iqr_capping(
    X_test,
    iqr_bounds
)

print("IQR outlier capping completed.")

IQR outlier capping completed.


In [17]:
nominal_columns = [
    "Brand",
    "Fuel_Type",
    "Transmission",
    "City",
    "Seller_Type"
]

ordinal_columns = [
    "Condition"
]

print("Nominal Columns:")
print(nominal_columns)

print("\nOrdinal Columns:")
print(ordinal_columns)

Nominal Columns:
['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']

Ordinal Columns:
['Condition']


In [18]:
condition_order = [
    "Poor",
    "Fair",
    "Good",
    "Very Good",
    "Excellent"
]

In [19]:
numeric_features = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Previous_Owners",
    "Accidents_Reported",
    "Service_Score"
]

nominal_features = [
    "Brand",
    "Fuel_Type",
    "Transmission",
    "City",
    "Seller_Type"
]

ordinal_features = [
    "Condition"
]

In [20]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [21]:
nominal_pipeline = Pipeline([
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

In [22]:
ordinal_pipeline = Pipeline([
    (
        "ordinal",
        OrdinalEncoder(
            categories=[condition_order],
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
    )
])

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "nominal",
            nominal_pipeline,
            nominal_features
        ),
        (
            "ordinal",
            ordinal_pipeline,
            ordinal_features
        )
    ]
)

In [24]:
X_train_processed = preprocessor.fit_transform(X_train_capped)

X_test_processed = preprocessor.transform(X_test_capped)

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape:", X_test_processed.shape)

Training processed shape: (256, 37)
Testing processed shape: (64, 37)


In [25]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

print("\nProcessed Feature Names:")
for feature in feature_names:
    print(feature)

Number of processed features: 37

Processed Feature Names:
numeric__Year
numeric__Mileage_Km
numeric__Engine_CC
numeric__Power_BHP
numeric__Previous_Owners
numeric__Accidents_Reported
numeric__Service_Score
nominal__Brand_Honda
nominal__Brand_Hyundai
nominal__Brand_Kia
nominal__Brand_Mahindra
nominal__Brand_Maruti
nominal__Brand_Renault
nominal__Brand_Skoda
nominal__Brand_Tata
nominal__Brand_Toyota
nominal__Brand_Volkswagen
nominal__Fuel_Type_CNG
nominal__Fuel_Type_Diesel
nominal__Fuel_Type_Electric
nominal__Fuel_Type_Petrol
nominal__Transmission_Automatic
nominal__Transmission_Manual
nominal__City_Ahmedabad
nominal__City_Bengaluru
nominal__City_Chandigarh
nominal__City_Delhi
nominal__City_Hyderabad
nominal__City_Jaipur
nominal__City_Kochi
nominal__City_Lucknow
nominal__City_Mumbai
nominal__City_Pune
nominal__Seller_Type_Certified Dealer
nominal__Seller_Type_Dealer
nominal__Seller_Type_Individual
ordinal__Condition


In [26]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

display(X_train_processed_df.head())

,numeric__Year,numeric__Mileage_Km,numeric__Engine_CC,numeric__Power_BHP,numeric__Previous_Owners,numeric__Accidents_Reported,numeric__Service_Score,nominal__Brand_Honda,nominal__Brand_Hyundai,nominal__Brand_Kia,...,nominal__City_Hyderabad,nominal__City_Jaipur,nominal__City_Kochi,nominal__City_Lucknow,nominal__City_Mumbai,nominal__City_Pune,nominal__Seller_Type_Certified Dealer,nominal__Seller_Type_Dealer,nominal__Seller_Type_Individual,ordinal__Condition
132,-0.486391,-0.225883,-0.322882,0.304485,-0.734379,-0.442634,-0.454105,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,3.0
317,0.725445,0.089371,-0.656431,-0.330444,-0.734379,1.477948,-0.614672,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0
234,-1.395268,1.115798,-0.120529,0.420784,-0.734379,-0.442634,-0.534389,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0
312,1.028404,-0.195076,0.479859,0.414498,0.433329,-0.442634,0.188165,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0
232,-1.092309,0.706680,0.786724,-0.437314,0.433329,-0.442634,0.107881,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0


In [27]:
X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

display(X_test_processed_df.head())

,numeric__Year,numeric__Mileage_Km,numeric__Engine_CC,numeric__Power_BHP,numeric__Previous_Owners,numeric__Accidents_Reported,numeric__Service_Score,nominal__Brand_Honda,nominal__Brand_Hyundai,nominal__Brand_Kia,...,nominal__City_Hyderabad,nominal__City_Jaipur,nominal__City_Kochi,nominal__City_Lucknow,nominal__City_Mumbai,nominal__City_Pune,nominal__Seller_Type_Certified Dealer,nominal__Seller_Type_Dealer,nominal__Seller_Type_Individual,ordinal__Condition
167,-0.486391,0.908398,-1.630394,-0.974803,-0.734379,-0.442634,-0.373821,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0
230,1.331363,-0.926995,0.784500,-0.226718,0.433329,-0.442634,-1.337226,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0
25,-1.092309,0.710326,-1.414699,-0.710773,-0.734379,-0.442634,1.552989,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0
63,-0.486391,0.866625,-1.630394,-0.776781,0.433329,-0.442634,1.071286,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,3.0
9,-1.395268,2.144119,-0.002675,0.100176,-0.734379,-0.442634,-0.775240,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0


In [28]:
test_processed = X_test_processed_df.copy()

test_processed["Resale_Price_Lakh"] = y_test.values

print("Processed Testing Dataset:")
print(test_processed.shape)

display(test_processed.head())

Processed Testing Dataset:
(64, 38)


,numeric__Year,numeric__Mileage_Km,numeric__Engine_CC,numeric__Power_BHP,numeric__Previous_Owners,numeric__Accidents_Reported,numeric__Service_Score,nominal__Brand_Honda,nominal__Brand_Hyundai,nominal__Brand_Kia,...,nominal__City_Jaipur,nominal__City_Kochi,nominal__City_Lucknow,nominal__City_Mumbai,nominal__City_Pune,nominal__Seller_Type_Certified Dealer,nominal__Seller_Type_Dealer,nominal__Seller_Type_Individual,ordinal__Condition,Resale_Price_Lakh
167,-0.486391,0.908398,-1.630394,-0.974803,-0.734379,-0.442634,-0.373821,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,1.62
230,1.331363,-0.926995,0.784500,-0.226718,0.433329,-0.442634,-1.337226,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0,8.35
25,-1.092309,0.710326,-1.414699,-0.710773,-0.734379,-0.442634,1.552989,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,2.11
63,-0.486391,0.866625,-1.630394,-0.776781,0.433329,-0.442634,1.071286,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,3.0,2.82
9,-1.395268,2.144119,-0.002675,0.100176,-0.734379,-0.442634,-0.775240,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0,1.20


In [29]:
train_processed = X_train_processed_df.copy()

train_processed["Resale_Price_Lakh"] = y_train.values

print("Processed Training Dataset:")
print(train_processed.shape)

display(train_processed.head())

Processed Training Dataset:
(256, 38)


,numeric__Year,numeric__Mileage_Km,numeric__Engine_CC,numeric__Power_BHP,numeric__Previous_Owners,numeric__Accidents_Reported,numeric__Service_Score,nominal__Brand_Honda,nominal__Brand_Hyundai,nominal__Brand_Kia,...,nominal__City_Jaipur,nominal__City_Kochi,nominal__City_Lucknow,nominal__City_Mumbai,nominal__City_Pune,nominal__Seller_Type_Certified Dealer,nominal__Seller_Type_Dealer,nominal__Seller_Type_Individual,ordinal__Condition,Resale_Price_Lakh
132,-0.486391,-0.225883,-0.322882,0.304485,-0.734379,-0.442634,-0.454105,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,3.0,4.26
317,0.725445,0.089371,-0.656431,-0.330444,-0.734379,1.477948,-0.614672,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,5.30
234,-1.395268,1.115798,-0.120529,0.420784,-0.734379,-0.442634,-0.534389,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,1.23
312,1.028404,-0.195076,0.479859,0.414498,0.433329,-0.442634,0.188165,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,7.09
232,-1.092309,0.706680,0.786724,-0.437314,0.433329,-0.442634,0.107881,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,2.69


In [30]:
scaled_numeric_features = [
    feature
    for feature in feature_names
    if feature.startswith("numeric__")
]

print("Mean of scaled numeric features:")
display(X_train_processed_df[scaled_numeric_features].mean())

print("\nStandard deviation of scaled numeric features:")
display(X_train_processed_df[scaled_numeric_features].std())

Mean of scaled numeric features:


,0
numeric__Year,-5.204170e-18
numeric__Mileage_Km,1.561251e-17
numeric__Engine_CC,2.753874e-17
numeric__Power_BHP,-7.208521e-16
numeric__Previous_Owners,5.984796e-17
numeric__Accidents_Reported,3.100818e-17
numeric__Service_Score,-5.800482e-18



Standard deviation of scaled numeric features:


,0
numeric__Year,1.001959
numeric__Mileage_Km,1.001959
numeric__Engine_CC,1.001959
numeric__Power_BHP,1.001959
numeric__Previous_Owners,1.001959
numeric__Accidents_Reported,1.001959
numeric__Service_Score,1.001959


In [31]:
print("Processed Training Dataset Information:")
print(train_processed.info())

print("\nProcessed Testing Dataset Information:")
print(test_processed.info())

Processed Training Dataset Information:
<class 'pandas.core.frame.DataFrame'>
Index: 256 entries, 132 to 102
Data columns (total 38 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   numeric__Year                          256 non-null    float64
 1   numeric__Mileage_Km                    256 non-null    float64
 2   numeric__Engine_CC                     256 non-null    float64
 3   numeric__Power_BHP                     256 non-null    float64
 4   numeric__Previous_Owners               256 non-null    float64
 5   numeric__Accidents_Reported            256 non-null    float64
 6   numeric__Service_Score                 256 non-null    float64
 7   nominal__Brand_Honda                   256 non-null    float64
 8   nominal__Brand_Hyundai                 256 non-null    float64
 9   nominal__Brand_Kia                     256 non-null    float64
 10  nominal__Brand_Mahindra              

In [32]:
print("Missing values in processed training data:")
print(train_processed.isnull().sum().sum())

print("\nMissing values in processed testing data:")
print(test_processed.isnull().sum().sum())

Missing values in processed training data:
0

Missing values in processed testing data:
0


In [33]:
print("========== FINAL VERIFICATION ==========")

print("Original Dataset:", df.shape)

print("Training Dataset:", train_processed.shape)

print("Testing Dataset:", test_processed.shape)

print("Training Missing Values:",
      train_processed.isnull().sum().sum())

print("Testing Missing Values:",
      test_processed.isnull().sum().sum())

========== FINAL VERIFICATION ==========
Original Dataset: (320, 15)
Training Dataset: (256, 38)
Testing Dataset: (64, 38)
Training Missing Values: 0
Testing Missing Values: 0


In [34]:
train_output = "Day12_Used_Car_Processed_Train.csv"
test_output = "Day12_Used_Car_Processed_Test.csv"

train_processed.to_csv(
    train_output,
    index=False
)

test_processed.to_csv(
    test_output,
    index=False
)

print("Files exported successfully!")

print("\nTraining file:")
print(train_output)

print("\nTesting file:")
print(test_output)

Files exported successfully!

Training file:
Day12_Used_Car_Processed_Train.csv

Testing file:
Day12_Used_Car_Processed_Test.csv


In [35]:
processed_full = pd.concat(
    [
        train_processed.assign(Dataset="Train"),
        test_processed.assign(Dataset="Test")
    ],
    axis=0
)

processed_full.to_csv(
    "Day12_Used_Car_Preprocessed_Dataset.csv",
    index=False
)

print("Final preprocessed dataset exported successfully!")

print("File:")
print("Day12_Used_Car_Preprocessed_Dataset.csv")

print("\nShape:")
print(processed_full.shape)

Final preprocessed dataset exported successfully!
File:
Day12_Used_Car_Preprocessed_Dataset.csv

Shape:
(320, 39)
